# Gemini Evaluation – Part II

End-to-end product retrieval, answer generation, and evaluation.

**Important:** the notebook uses `utils.py` for all Gemini calls. There is no second Gemini client in this notebook.

In [54]:
import os
import importlib

# Put your NEW Gemini API key here, or set it in Windows/Jupyter beforehand.
# Do not paste a real key into a notebook that you plan to share.
os.environ["GEMINI_API_KEY"] = "apikey"

import utils
importlib.reload(utils)

print("utils loaded successfully")
print("Model:", utils.DEFAULT_MODEL)

utils loaded successfully
Model: gemini-3.6-flash


In [55]:
# Optional connection test
# If Gemini is temporarily busy, utils.py will retry automatically.

test = utils.test_gemini_connection()
print("Connection test:", test)

Gemini request (attempt 1/5)

Gemini returned an empty response.
Response type: GenerateContentResponse
Candidates: [Candidate(
  content=Content(),
  finish_reason=<FinishReason.MAX_TOKENS: 'MAX_TOKENS'>,
  index=0
)]

Gemini test response:
None
Connection test: None


## Run through the end-to-end system

In [56]:
customer_msg = """
tell me about the smartx pro phone and the fotosnap camera, the dslr one.
Also, what TVs or TV related products do you have?
"""

# Step 1: find products/categories from the customer question
products_by_category = utils.get_products_from_query(customer_msg)
print("PRODUCTS FROM QUERY:")
print(products_by_category)

Gemini request (attempt 1/5)
PRODUCTS FROM QUERY:
[
  {
    "category": "Smartphones and Accessories",
    "products": [
      "SmartX ProPhone"
    ]
  },
  {
    "category": "Cameras and Camcorders",
    "products": [
      "FotoSnap DSLR Camera"
    ]
  },
  {
    "category": "Televisions and Home Theater Systems",
    "products": [
      "CineView 4K TV",
      "CineView 8K TV",
      "CineView OLED TV",
      "SoundMax Home Theater",
      "SoundMax Soundbar"
    ]
  }
]


In [57]:
# Step 2: convert the model string to a Python list
category_and_product_list = utils.read_string_to_list(products_by_category)

print("CATEGORY AND PRODUCT LIST:")
print(category_and_product_list)

CATEGORY AND PRODUCT LIST:
[{'category': 'Smartphones and Accessories', 'products': ['SmartX ProPhone']}, {'category': 'Cameras and Camcorders', 'products': ['FotoSnap DSLR Camera']}, {'category': 'Televisions and Home Theater Systems', 'products': ['CineView 4K TV', 'CineView 8K TV', 'CineView OLED TV', 'SoundMax Home Theater', 'SoundMax Soundbar']}]


In [58]:
# Step 3: get the detailed product information for the selected products
product_info = utils.get_mentioned_product_info(category_and_product_list)

print("PRODUCT INFORMATION:")
print(product_info)

PRODUCT INFORMATION:
{'Smartphones and Accessories': {'SmartX ProPhone': {'category': 'Smartphones and Accessories', 'description': 'A powerful smartphone with advanced camera features.', 'features': ['12MP dual camera', '5G wireless', '128GB storage', '6.1-inch display'], 'price': '$899.99'}}, 'Cameras and Camcorders': {'FotoSnap DSLR Camera': {'category': 'Cameras and Camcorders', 'description': 'A DSLR camera for capturing photos and videos.', 'features': ['1080p video', '3-inch LCD', '24.2MP sensor', 'interchangeable lenses'], 'price': '$599.99'}}, 'Televisions and Home Theater Systems': {'CineView 4K TV': {'category': 'Televisions and Home Theater Systems', 'description': 'A 4K TV with vibrant colors and smart features.', 'features': ['55-inch display', '4K resolution', 'HDR', 'Smart TV'], 'price': '$599.00'}, 'CineView 8K TV': {'category': 'Televisions and Home Theater Systems', 'description': 'A stunning 8K TV.', 'features': ['65-inch display', '8K resolution', 'HDR', 'Smart TV'

In [59]:
# Step 4: generate the final customer-service answer
assistant_answer = utils.answer_user_msg(
    user_msg=customer_msg,
    product_info=product_info,
)

print("ASSISTANT ANSWER:")
print(assistant_answer)

Gemini request (attempt 1/5)
ASSISTANT ANSWER:
Hello! I'd be happy to help you with information about our products. Here are the details you requested:

### **SmartX ProPhone**
* **Description:** A powerful smartphone with advanced camera features.
* **Features:** 6.1-inch display, 12MP dual camera, 5G wireless, and 128GB storage.
* **Price:** $899.99

---

### **FotoSnap DSLR Camera**
* **Description:** A DSLR camera designed for capturing photos and videos.
* **Features:** 24.2MP sensor, 3-inch LCD screen, 1080p video recording, and interchangeable lenses.
* **Price:** $599.99

---

### **TVs & TV-Related Products**

We offer several televisions and home theater audio options:

#### **Televisions:**
1. **CineView 4K TV**
   * **Description:** A 4K TV with vibrant colors and smart features.
   * **Features:** 55-inch display, 4K resolution, HDR, Smart TV capabilities.
   * **Price:** $599.00

2. **CineView 8K TV**
   * **Description:** A stunning 8K TV.
   * **Features:** 65-inch disp

## Evaluate the LLM answer with a rubric

In [60]:
cust_prod_info = {
    "customer_msg": customer_msg,
    "context": product_info,
}

In [61]:
def eval_with_rubric(test_set, assistant_answer):
    cust_msg = test_set["customer_msg"]
    context = test_set["context"]
    completion = assistant_answer

    system_message = """
You evaluate how well a customer service agent answers a user question using the supplied context.
Return a concise evaluation.
"""

    user_message = f"""
[BEGIN DATA]
[Question]: {cust_msg}
[Context]: {context}
[Submission]: {completion}
[END DATA]

Answer:
- Is the Assistant response based only on the context provided? (Y or N)
- Does the answer include information that is not provided in the context? (Y or N)
- Is there any disagreement between the response and the context? (Y or N)
- Count how many questions the user asked. (number)
- For each question, is there a corresponding answer? Question 1: Y/N, Question 2: Y/N, etc.
- Of the questions asked, how many were addressed? (number)
"""

    return utils.get_completion_from_messages([
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ], temperature=0, max_tokens=500)

In [62]:
evaluation_output = eval_with_rubric(cust_prod_info, assistant_answer)
print(evaluation_output)

Gemini request (attempt 1/5)

Gemini ClientError:
429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 51.971447333s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'g

## Evaluate against an ideal / expert answer

In [63]:
test_set_ideal = {
    "customer_msg": customer_msg,
    "ideal_answer": """
Of course! The SmartX ProPhone is a powerful smartphone with advanced camera features.
For instance, it has a 12MP dual camera, 5G wireless, 128GB storage, and a 6.1-inch display.
The price is $899.99.

The FotoSnap DSLR Camera is great for capturing stunning photos and videos.
Features include 1080p video, 3-inch LCD, a 24.2MP sensor, and interchangeable lenses.
The price is $599.99.

For TVs and TV-related products, we offer CineView 4K TV, CineView 8K TV,
CineView OLED TV, SoundMax Home Theater, and SoundMax Soundbar.
The TV products offer HDR and Smart TV. The home theater products include Bluetooth.
""",
}

In [64]:
def eval_vs_ideal(test_set, assistant_answer):
    cust_msg = test_set["customer_msg"]
    ideal = test_set["ideal_answer"]
    completion = assistant_answer

    system_message = """
You evaluate a submitted answer against an expert answer.
Return one letter only: A, B, C, D, or E.
"""

    user_message = f"""
[BEGIN DATA]
[Question]: {cust_msg}
[Expert]: {ideal}
[Submission]: {completion}
[END DATA]

(A) Submission is a subset of the expert and fully consistent.
(B) Submission is a superset of the expert and fully consistent.
(C) Submission contains all the same details as the expert.
(D) There is a factual disagreement.
(E) Differences do not matter for factuality.

Return only A, B, C, D, or E.
"""

    return utils.get_completion_from_messages([
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ], temperature=0, max_tokens=10)

In [67]:
print("Assistant answer:\n")
print(assistant_answer)

print("\nIdeal evaluation:")
print(eval_vs_ideal(test_set_ideal, assistant_answer))

Assistant answer:

Hello! I'd be happy to help you with information about our products. Here are the details you requested:

### **SmartX ProPhone**
* **Description:** A powerful smartphone with advanced camera features.
* **Features:** 6.1-inch display, 12MP dual camera, 5G wireless, and 128GB storage.
* **Price:** $899.99

---

### **FotoSnap DSLR Camera**
* **Description:** A DSLR camera designed for capturing photos and videos.
* **Features:** 24.2MP sensor, 3-inch LCD screen, 1080p video recording, and interchangeable lenses.
* **Price:** $599.99

---

### **TVs & TV-Related Products**

We offer several televisions and home theater audio options:

#### **Televisions:**
1. **CineView 4K TV**
   * **Description:** A 4K TV with vibrant colors and smart features.
   * **Features:** 55-inch display, 4K resolution, HDR, Smart TV capabilities.
   * **Price:** $599.00

2. **CineView 8K TV**
   * **Description:** A stunning 8K TV.
   * **Features:** 65-inch display, 8K resolution, HDR, Sma

In [66]:
# Negative test
assistant_answer_2 = "life is like a box of chocolates"
print(eval_vs_ideal(test_set_ideal, assistant_answer_2))

Gemini request (attempt 1/5)

Gemini ClientError:
429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 49.767372743s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'g